# Astra v3 — Security-Issue Detection Training (T4, ~2 hours)

Trains a cyber-focused model on the **NSL-KDD intrusion detection corpus** to detect security issues in system telemetry.

**Pipeline**:
1. **Build cyber BPE tokenizer** (8192 vocab) from NSL-KDD train split
2. **Pre-train astra5m_cyber** on 120k alert rows → downloads as `astra_cyber_pretrain_final.npz`
3. **Fine-tune on held-out test21** (11.8k rows, 17 novel attack classes) → downloads as `astra_cyber_finetune_final.npz`

Each checkpoint has a **unique name** so nothing in Downloads/Drive gets confused.

Runtime: **Settings → Runtime → GPU (T4)**, Internet ON. Run cells top to bottom.

In [ ]:
import torch
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))

**Diagnostic — what exists on this VM** (run whenever unsure):

In [ ]:
import os
paths = [
    '/content/astra',
    '/content/astra/tokenizer/artifacts/cyber_bpe.json',
    '/content/runs/cyber_pretrain/final.npz',
    '/content/runs/cyber_finetune/resumed/final.npz',
    '/content/drive/MyDrive/astra_checkpoints/astra_cyber_pretrain_final.npz',
    '/content/drive/MyDrive/astra_checkpoints/astra_cyber_finetune_final.npz',
]
for p in paths:
    ok = os.path.exists(p)
    sz = os.path.getsize(p) if ok else 0
    print(f"{'YES' if ok else 'NO '}  {sz:>12,}  {p}")

In [ ]:
import os, subprocess
REPO = '/content/astra'
if os.path.isdir(REPO):
    subprocess.run(f'rm -rf {REPO}', shell=True, check=True)
subprocess.run(f'git clone --depth 1 https://github.com/anoneurx/astra.git {REPO}', shell=True, check=True)
assert os.path.isdir(REPO), 'clone failed - check Internet is ON and retry'
os.chdir(REPO)
print('repo ready at', os.getcwd())

**Mount Google Drive** — persistent backup in case the VM recycles:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/astra_checkpoints', exist_ok=True)
print('Drive ready')

---
## STAGE 0 — Build cyber BPE tokenizer (one-time, ~30 sec)
Produces `tokenizer/artifacts/cyber_bpe.json` (gitignored).

In [ ]:
import os, subprocess, sys
os.chdir('/content/astra')
sys.path.insert(0, '/content/astra/python')
if not os.path.exists('tokenizer/artifacts/cyber_bpe.json'):
    subprocess.run('python tokenizer/train_tokenizer.py --config configs/tokenizer_cyber.json', shell=True, check=True)
import json
tok = json.load(open('tokenizer/artifacts/cyber_bpe.json'))
print('tokenizer vocab:', len(tok.get('merges', [])) + tok.get('vocab_size', 0) if 'vocab_size' in tok else 'see artifact')
# Quick validation
from astra.tokenizer import load_tokenizer
t = load_tokenizer('tokenizer/artifacts/cyber_bpe.json')
sample = "ALERT duration=0 protocol=tcp service=http flag=SF srcbytes=181 dstbytes=5450 | EARLY_WARNING normal HARDNESS=21"
ids = t.encode(sample)
print('sample encode:', ids[:20], '... (len=', len(ids), ')')
print('roundtrip:', t.decode(ids) == sample)

---
## STAGE 1 — Cyber pre-train (50,000 steps, ~50 min on T4)
Ends with **`[step 50000]`** and auto-downloads as **`astra_cyber_pretrain_final.npz`**.

**Config**: `configs/astra5m_cyber.json` — 5.1M params, 6-layer, d_model=256, seq_len=128.
Training tokens: ~12M (120k rows × ~100 tokens/row). One epoch ≈ 6,000 steps at batch_seq=16.
50k steps ≈ 8 epochs — enough for convergence on this corpus.

In [ ]:
import os, subprocess, shutil
os.chdir('/content/astra')
if not os.path.exists('tokenizer/artifacts/cyber_bpe.json'):
    subprocess.run('python tokenizer/train_tokenizer.py --config configs/tokenizer_cyber.json', shell=True, check=True)
!mkdir -p /content/runs
# Override steps to ~50k for ~2hr budget; other params from config
!python training/gpu_train.py --config configs/astra5m_cyber.json \
  --steps 50000 --out /content/runs/cyber_pretrain \
  --cache-dir /content/cache 2>&1 | tail -15
SRC = '/content/runs/cyber_pretrain/final.npz'
assert os.path.exists(SRC), 'PRETRAIN NOT PRODUCED - training failed'
# copy to unique names, then download
shutil.copy(SRC, '/content/astra_cyber_pretrain_final.npz')
shutil.copy(SRC, '/content/drive/MyDrive/astra_checkpoints/astra_cyber_pretrain_final.npz')
print('PRETRAIN ok:', os.path.getsize('/content/astra_cyber_pretrain_final.npz'), 'bytes')
from google.colab import files
files.download('/content/astra_cyber_pretrain_final.npz')

---
## STAGE 2 — Fine-tune on held-out test21 (10,000 steps, ~10 min)
**Only run after STAGE 1 finished.** Warm-starts from pre-train final.
The test21 split contains 17 attack classes **never seen during pre-train** (mscan, apache2, saint, etc.) — this measures true generalization.
Ends with **`[step 10000]`** and auto-downloads as **`astra_cyber_finetune_final.npz`**.

In [ ]:
import os, subprocess, shutil
os.chdir('/content/astra')
if not os.path.exists('tokenizer/artifacts/cyber_bpe.json'):
    subprocess.run('python tokenizer/train_tokenizer.py --config configs/tokenizer_cyber.json', shell=True, check=True)
PRETRAIN = '/content/runs/cyber_pretrain/final.npz'
assert os.path.exists(PRETRAIN), 'NO PRETRAIN FINAL - run STAGE 1 first'
# Fine-tune on test21 (held-out, novel attacks) - create a temp config override
import json
cfg = json.load(open('configs/astra5m_cyber.json'))
cfg['data']['train'] = 'datasets/cyber/test21.txt'
cfg['data']['val'] = 'datasets/cyber/val.txt'
cfg['training']['max_steps'] = 10000
cfg['training']['val_every'] = 500
cfg['out_dir'] = 'checkpoints/astra5m_cyber_finetune'
ft_cfg_path = '/content/astra5m_cyber_finetune.json'
json.dump(cfg, open(ft_cfg_path, 'w'), indent=2)
print('Fine-tune config written to', ft_cfg_path)
!python training/gpu_train.py --config {ft_cfg_path} \
  --steps 10000 --out /content/runs/cyber_finetune \
  --resume {PRETRAIN} --reset-step \
  --cache-dir /content/cache 2>&1 | tail -15
SRC = '/content/runs/cyber_finetune/resumed/final.npz'
assert os.path.exists(SRC), 'FINETUNE FINAL NOT PRODUCED - training failed'
# copy to unique names, then download
shutil.copy(SRC, '/content/astra_cyber_finetune_final.npz')
shutil.copy(SRC, '/content/drive/MyDrive/astra_checkpoints/astra_cyber_finetune_final.npz')
print('FINETUNE ok:', os.path.getsize('/content/astra_cyber_finetune_final.npz'), 'bytes')
from google.colab import files
files.download('/content/astra_cyber_finetune_final.npz')

---
## STAGE 3 — Quick evaluation on test21 (optional, ~30 sec)
Runs a single validation pass on the held-out test21 set to confirm the model detects novel attacks.

In [ ]:
import os, json, subprocess
os.chdir('/content/astra')
CKPT = '/content/runs/cyber_finetune/resumed/final.npz'
assert os.path.exists(CKPT), 'No finetune checkpoint - run STAGE 2 first'
# Use the existing evaluation harness (loads NumPy checkpoint into torch model)
result = subprocess.run(
    'python evaluation/evaluate.py --checkpoint ' + CKPT + ' --config configs/astra5m_cyber.json',
    shell=True, capture_output=True, text=True
)
print(result.stdout[-2000:] if result.stdout else 'no stdout')
if result.stderr:
    print('STDERR:', result.stderr[-1000:])

---
## Download / re-download (optional)
Pulls from the VM **and** Drive with the distinct names. Only files that exist are downloaded.

In [ ]:
from google.colab import files
import os, shutil
cands = {
    'astra_cyber_pretrain_final.npz': [
        '/content/runs/cyber_pretrain/final.npz',
        '/content/drive/MyDrive/astra_checkpoints/astra_cyber_pretrain_final.npz',
    ],
    'astra_cyber_finetune_final.npz': [
        '/content/runs/cyber_finetune/resumed/final.npz',
        '/content/drive/MyDrive/astra_checkpoints/astra_cyber_finetune_final.npz',
    ],
}
for name, paths in cands.items():
    src = next((p for p in paths if os.path.exists(p)), None)
    if src is None:
        print(f'NO FILE: {name} (neither VM nor Drive has it)')
        continue
    dst = f'/content/{name}'
    shutil.copy(src, dst)
    print(f'DOWNLOADING {name} ({os.path.getsize(dst):,} bytes) from {src}')
    files.download(dst)

In [ ]:
import hashlib
for name in ['astra_cyber_pretrain_final.npz', 'astra_cyber_finetune_final.npz']:
    p = f'/content/{name}'
    if os.path.exists(p):
        h = hashlib.md5(open(p, 'rb').read()).hexdigest()[:12]
        print(name, os.path.getsize(p), 'md5', h)
    else:
        print(name, 'NOT PRESENT')